In [1]:
from __future__ import annotations

import glob
import io
from pathlib import Path
from typing import Tuple

import pandas as pd

# --- Config ---
# GRID_ROOT = Path("results/bbob/")
GRID_ROOT = Path("results/bbob/")
SPLITS = ["LPO", "LIO", "RANDOM"]
FILENAME_GLOB = "res_tailpenalty_*.csv"

# If you want shorter section names, set this to True.
SHORTEN_SECTION_NAMES = False

def _extract_table_block(lines: list[str], stat: str, substat: str) -> pd.DataFrame:
    """Extract a rectangular CSV table that starts at:
        <stat>
        <substat>
        <header, comma-separated>
        <rows...>
        <blank line>
    Returns a DataFrame indexed by the first column (e.g. 'Problem Group').
    """
    start = None
    for i in range(len(lines) - 1):
        if lines[i].strip() == stat and lines[i + 1].strip() == substat:
            start = i + 2
            break
    if start is None:
        raise ValueError(f"Could not find section '{stat} -> {substat}'")

    header = lines[start].strip()
    if not header:
        raise ValueError(f"Empty header for section '{stat} -> {substat}'")

    data_lines: list[str] = []
    for j in range(start + 1, len(lines)):
        line = lines[j].strip()
        if line == "":
            break
        data_lines.append(lines[j])

    block = "\n".join([header] + data_lines)
    df = pd.read_csv(io.StringIO(block))
    if df.shape[1] < 2:
        raise ValueError(
            f"Parsed table for '{stat} -> {substat}' looks wrong: columns={df.columns.tolist()}"
        )

    df = df.set_index(df.columns[0])
    return df

def read_mean_median_as_tables(csv_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return (mean_as, median_as) tables from a result CSV."""
    text = csv_path.read_text()
    lines = text.splitlines()
    mean_as = _extract_table_block(lines, "Mean", "AS")
    median_as = _extract_table_block(lines, "Median", "AS")
    p90_as = _extract_table_block(lines, "P90", "AS")
    return mean_as, median_as, p90_as

def _maybe_shorten(section: str) -> str:
    if not SHORTEN_SECTION_NAMES:
        return section
    return section.replace("tail1_", "").replace("tail0_", "").replace("dual1_", "dual=")

def build_sectioned_csv_for_split(split: str) -> Path:
    # Discover all matching files
    pattern = str(GRID_ROOT / "*" / split / FILENAME_GLOB)
    csv_paths = sorted(Path(p) for p in glob.glob(pattern))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found for split={split} under {pattern}")

    # Precompute per-grid counts (so we can disambiguate section names if needed)
    grid_counts: dict[Path, int] = {}
    for p in csv_paths:
        grid_dir = p.parents[2]  # .../_grid_tmp/<grid>/<split>/tail_dual_head/<file>
        grid_counts[grid_dir] = grid_counts.get(grid_dir, 0) + 1

    out_path = Path(f"AS_mean_median_p90__{split}__ALL_RUNS.csv")
    bad: list[tuple[Path, str]] = []
    wrote = 0

    with out_path.open("w", encoding="utf-8", newline="\n") as f:
        for csv_path in csv_paths:
            grid_dir = csv_path.parents[1]
            section = _maybe_shorten(grid_dir.name)
            # if grid_counts[grid_dir] > 1:
            section = f"{section} -- {csv_path.stem}"

            try:
                mean_as, median_as, p90_as = read_mean_median_as_tables(csv_path)
            except Exception as e:
                bad.append((csv_path, str(e)))
                continue

            # Desired format:
            # {parameter set}
            # Mean
            # {table}
            # Median
            # {table}
            # if (mean_as.iloc[-1, -1] < 30) & (median_as.iloc[-1, -1] < 1.3):
            f.write(f"{section}\n")
            f.write("Mean\n")
            mean_as.to_csv(f)
            f.write("Median\n")
            median_as.to_csv(f)
            f.write("P90\n")
            p90_as.to_csv(f)
            f.write("\n")
            wrote += 1

    if bad:
        print(f"\n[WARN] Failed to parse {len(bad)} file(s) for split={split}:")
        for p, msg in bad[:20]:
            print(f"  - {p}: {msg}")
        if len(bad) > 20:
            print(f"  ... and {len(bad) - 20} more")

    print(f"Wrote {out_path} with {wrote} parameter set(s) (each includes Mean+Median+P90 tables).")
    return out_path

# --- Run for all splits ---
outs = {split: build_sectioned_csv_for_split(split) for split in SPLITS}
outs

Wrote AS_mean_median_p90__LPO__ALL_RUNS.csv with 90 parameter set(s) (each includes Mean+Median+P90 tables).
Wrote AS_mean_median_p90__LIO__ALL_RUNS.csv with 90 parameter set(s) (each includes Mean+Median+P90 tables).
Wrote AS_mean_median_p90__RANDOM__ALL_RUNS.csv with 90 parameter set(s) (each includes Mean+Median+P90 tables).


{'LPO': PosixPath('AS_mean_median_p90__LPO__ALL_RUNS.csv'),
 'LIO': PosixPath('AS_mean_median_p90__LIO__ALL_RUNS.csv'),
 'RANDOM': PosixPath('AS_mean_median_p90__RANDOM__ALL_RUNS.csv')}

In [ ]:
from __future__ import annotations

from pathlib import Path

# Inputs (edit if you want different paths/order)
FILES = {
    "LPO": Path("AS_mean_median_p90__LPO__ALL_RUNS.csv"),
    "LIO": Path("AS_mean_median_p90__LIO__ALL_RUNS.csv"),
    "RANDOM": Path("AS_mean_median_p90__RANDOM__ALL_RUNS.csv"),
}
OUT = Path("AS_mean_median_p90__MERGED__ALL_RUNS.csv")
def parse_sectioned_csv(path: Path) -> dict[str, dict[str, list[str]]]:
    """Parse: section -> {'Mean': [csv lines], 'Median': [csv lines]} preserving raw CSV text."""
    lines = path.read_text(encoding="utf-8").splitlines()
    i, n = 0, len(lines)
    out: dict[str, dict[str, list[str]]] = {}

    def skip_blanks(j: int) -> int:
        while j < n and lines[j].strip() == "":
            j += 1
        return j

    while True:
        i = skip_blanks(i)
        if i >= n:
            break
        section = lines[i].strip()
        i += 1

        i = skip_blanks(i)
        if i >= n or lines[i].strip() != "Mean":
            raise ValueError(f"{path}: expected 'Mean' after section '{section}'")
        i += 1
        mean: list[str] = []
        while i < n and lines[i].strip() != "Median":
            mean.append(lines[i])
            i += 1

        if i >= n or lines[i].strip() != "Median":
            raise ValueError(f"{path}: missing 'Median' for section '{section}'")
        i += 1
        median: list[str] = []
        while i < n and lines[i].strip() != "P90":
            median.append(lines[i])
            i += 1

        if i >= n or lines[i].strip() != "P90":
            raise ValueError(f"{path}: missing 'P90' for section '{section}'")
        i += 1
        p90: list[str] = []
        while i < n and lines[i].strip() != "":
            p90.append(lines[i])
            i += 1

        out[section] = {"Mean": mean, "Median": median, "P90": p90}
    return out

parsed = {split: parse_sectioned_csv(p) for split, p in FILES.items()}
order = list(parsed[next(iter(parsed))].keys())  # keep order from the first file

missing = []
for section in order:
    for split in FILES:
        if section not in parsed[split]:
            missing.append((section, split))
if missing:
    print("[WARN] Missing sections:")
    for section, split in missing[:20]:
        print(f"  - {section} missing in {split}")
    if len(missing) > 20:
        print(f"  ... and {len(missing) - 20} more")

with OUT.open("w", encoding="utf-8", newline="\n") as f:
    for section in order:
        violate = False
        for split in FILES:

            if split == "LPO":
                threshold_mean = 300
                threshold_median = 1.8
                threshold_p90 = 10
            else:
                threshold_mean = 300
                threshold_median = 1.8
                threshold_p90 = 7

            # Separate LIO check
            # if split == "LPO":
            #     threshold_mean = 1000
            #     threshold_median = 1000
            #     threshold_p90 = 1000
            # elif split == "LIO":
            #     threshold_mean = 15
            #     threshold_median = 1.5
            #     threshold_p90 = 6
            # else:
            #     threshold_mean = 1000
            #     threshold_median = 1000
            #     threshold_p90 = 1000
            
            if section not in parsed[split]:
                continue

            # print(parsed[split][section]["Mean"][-1].split(',')[-1])
            # raise
            violate = float(parsed[split][section]["Mean"][-1].split(',')[-1]) >= threshold_mean or \
                    float(parsed[split][section]["Median"][-1].split(',')[-1]) >= threshold_median or \
                    float(parsed[split][section]["P90"][-1].split(',')[-1]) >= threshold_p90 or \
                    violate
            
        if violate:
            continue

        f.write(section + "\n")
        for split in FILES:
            
            if section not in parsed[split]:
                continue

            f.write(split + "\n")
            f.write("Mean\n")
            f.write("\n".join(parsed[split][section]["Mean"]) + "\n")
            f.write("Median\n")
            f.write("\n".join(parsed[split][section]["Median"]) + "\n")
            f.write("P90\n")
            f.write("\n".join(parsed[split][section]["P90"]) + "\n")
        f.write("\n")

print(f"Wrote {OUT}")

Wrote AS_mean_median_p90__MERGED__ALL_RUNS.csv
